# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via the FAIR<sup>2</sup> Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and explore its content using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL defining the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}\n")
print(f"Data Collection Protocol: {getattr(metadata, 'dataCollection', 'N/A')}")

## 2. Data Overview
Review the available record sets, their fields, and `@id`s. We will print all record sets, then select one for analysis, referencing all entities (record sets, fields, columns) by their `@id` as per Croissant best practices.

In [ ]:
# List all available record sets and their @id
record_sets = list(dataset.record_sets)
print(f"Total Record Sets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record Set Label: {record_set.label if hasattr(record_set, 'label') else getattr(record_set, 'name', 'N/A')}")
    print(f"  @id         : {record_set.id}")
    print(f"  Description : {getattr(record_set, 'description', 'N/A')}")
    # Fields in the record set
    fields = list(record_set.fields)
    print(f"  Fields ({len(fields)}):")
    for field in fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {getattr(field, 'data_type', None)})")
    print("")

## 3. Data Extraction
We select a record set for analysis. All references are by `@id`. Data is loaded into a pandas DataFrame.

**⚠️ If you are unsure which record set to use, check the output above and select a record set that contains the main tabular data (likely large and with numeric fields).**

Below, edit the `target_record_set_id` variable to the `@id` of the record set you want to work with (as printed above).

In [ ]:
# Set the target record set by its `@id` for extraction (edit if needed)
target_record_set_id = None
# Try to auto-select if only one tabular main RecordSet
if len(record_sets) == 1:
    target_record_set_id = record_sets[0].id
else:
    # Otherwise, set manually, e.g.,
    # target_record_set_id = 'cr:MainTable'
    pass

assert target_record_set_id is not None, 'You must set target_record_set_id to the @id of your record set!'

# List all record set @id's if needed
print(f"Record sets found: {[rs.id for rs in record_sets]}")

# Load all records from the selected record set
records = list(dataset.records(record_set=target_record_set_id))
df = pd.DataFrame(records)
print(f"Extracted {len(df)} records from record set @id: {target_record_set_id}")
print("Columns (@id):", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filtering, normalization, categorization.

- Pick a numeric field by its `@id` from the DataFrame columns above.
- Optionally select a field for grouping/aggregation.

Set the variables `numeric_field_id` and `group_field_id` as appropriate for your use case.

In [ ]:
# Set the numeric field @id to use (edit as per your data)
numeric_field_id = None  # Example: 'cr:Field:coverage_percentage'
# Optionally set a field @id to group by (edit as per your data)
group_field_id = None  # Example: 'cr:Field:sample_id'

# Try to auto-select a numeric column if not set
if numeric_field_id is None:
    # Try to pick the first float or int-like column
    for col in df.columns:
        # Try to convert to float to test column type
        try:
            df[col].astype(float)
            numeric_field_id = col
            break
        except Exception:
            continue

assert numeric_field_id in df.columns, 'Set numeric_field_id to a numeric field @id from your DataFrame columns.'

# Filter records (example: values > 10)
threshold = 10
df_numeric = pd.to_numeric(df[numeric_field_id], errors='coerce')
filtered_df = df[df_numeric > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
print(filtered_df.head())

# Normalize selected field
mean_val = df_numeric.mean()
std_val = df_numeric.std()
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - mean_val) / std_val
print(f"First normalized {numeric_field_id} values:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group and aggregate
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field, and its normalized version. You may need to install `matplotlib` and `seaborn` if not available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True, color='dodgerblue')
plt.title(f"Distribution of {numeric_field_id} (Filtered)")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If group_field_id available and used
if group_field_id is not None and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR<sup>2</sup> mass spectrometry dataset using `mlcroissant`, referencing all entities (record sets, fields) by their `@id`s for reproducibility and interoperability.

We reviewed the data schema, extracted records, performed basic filtering, normalization, and visualized selected fields. Further analyses (statistical comparisons, advanced modeling, or domain-specific processing) can be performed depending on your research or data curation goals.

**References:**

- [FAIR2 Dataset on SenScience](https://sen.science/doi/10.71728/senscience.vq4a-28xa/)
- [`mlcroissant` documentation](https://github.com/mlcommons/croissant)
- [Croissant Standard](https://mlcommons.org/croissant/)
